# Memory and Context Engineering for AI Agents with Oracle AI Database, Langchain and Tavily

[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/memory_context_engineering_agents.ipynb)

--------



In this notebook, you'll learn how to engineer memory systems that give AI agents the ability to remember, learn, and adapt across conversations. 
Moving beyond simple RAG, we implement a complete **Memory Manager** with six distinct memory types—each serving a specific cognitive function.



## What You'll Build

| Memory Type | Purpose | Storage |
|-------------|---------|---------|
| **Conversational** | Chat history per thread | SQL Table |
| **Knowledge Base** | Searchable documents & facts | Vector-Enabled SQL Table |
| **Workflow** | Learned action patterns | Vector-Enabled SQL Table |
| **Toolbox** | Dynamic tool definitions | Vector-Enabled SQL Table |
| **Entity** | People, places, systems extracted from context | Vector-Enabled SQL Table |
| **Summary** | Compressed context for long conversations | Vector-Enabled SQL Table |
| **Tool Log** | Offloaded tool call outputs (experimental memory) | SQL Table |


## Key Concepts Covered

- **Memory Engineering**: Design patterns for agent memory systems
- **Context Engineering**: Techniques for optimizing what goes into the LLM context
- **Context Window Management**: Monitor usage and compact context when needed
- **Just-in-Time Retrieval**: Compact summaries with on-demand expansion
- **Dynamic Tool Calling**: Semantic tool discovery and execution
- **Entity Extraction**: LLM-powered entity recognition and storage



## Prerequisites

- Python 3.10+
- Oracle AI Database (local Docker or cloud)
- OpenAI API key
- Tavily API key

## By the End
You'll have a reusable `MemoryManager` and a turn-level agent harness (`call_agent`) that demonstrates how modern AI agents maintain context, learn from interactions, and manage information across sessions.

In [ ]:
! pip install -qU langchain-oracledb sentence-transformers langchain-openai langchain tavily-python datasets

# Part 1: Local Installation of Oracle AI Database via Docker [Memory Core]

--------

This section walks you through setting up **Oracle AI Database 26ai** locally using Docker. Oracle AI Database is a converged database that combines relational, document, graph, and vector data in a single engine—making it ideal for AI applications that need semantic search, embeddings storage, and vector similarity queries.

**What you'll do:**
1. Pull and run the Oracle Database Docker container
2. Establish a connection from Python using `oracledb`
3. Create a dedicated user for vector operations

This local setup gives you a fully functional Oracle database for development and testing without needing cloud infrastructure.

### Step 1: Installing Oracle AI Database via Docker

For this notebook we will be using a local installation of [Oracle AI Database](https://www.oracle.com/database/free/get-started/)

1. Install & start Docker. Docker Desktop (Mac/Windows) or Docker Engine (Linux). Make sure it’s running.
    - If installed with Docker Enginer, run from terminal ```open /Applications/Docker.app```
2. We are going to pull the [docker image](https://container-registry.oracle.com/ords/f?p=113:4:13936724845291:::4:P4_REPOSITORY,AI_REPOSITORY,AI_REPOSITORY_NAME,P4_REPOSITORY_NAME,P4_EULA_ID,P4_BUSINESS_AREA_ID:1863,1863,Oracle%20Database%20Free,Oracle%20Database%20Free,1,0&cs=3cVNH02fFYhB723ODpNnr0JZI1S7Z64nRyL_zC1Ls5BSVLafGsOLMFvFoPhn8JeeB8tXPhkfFKH8-dkrL_z3_0g)
3. Run a container with oracle image

    ```
      docker run -d \
        --name oracle-free \
        -p 1521:1521 -p 5500:5500 \
        -e ORACLE_PWD=OraclePwd_2025 \
        -v $HOME/oracle/full_data:/opt/oracle/oradata \
        container-registry.oracle.com/database/free:latest

    ```

> 🚫 **Troubleshoot**  
> If you see the error:  
> *`docker: Error response from daemon: Conflict. The container name "/oracle-full" is already in use by container ... You have to remove (or rename) that container to be able to reuse that name.`*  
>
> 🧩 **Fix:**  
> - Remove the existing container:  
>   ```bash
>   docker rm oracle-free
>   ```  
> - Then re-run your Docker command from **Step 3** to start a new container.


### Step 2: One-Click Database Setup

The cell below handles **everything automatically**:
- ✅ Checks if Docker is running
- ✅ Checks if Oracle container exists and is healthy
- ✅ Waits for database to be ready (with progress indicator)
- ✅ Fixes the listener for ARM Macs (Apple Silicon)
- ✅ Creates the VECTOR user with proper privileges
- ✅ Tests the connection

**Just run the cell below and wait for the ✅ success message!**


In [ ]:
import subprocess
import time
import sys

def setup_oracle_database(container_name="oracle-free", vector_password="VectorPwd_2025"):
    """
    Complete Oracle Database setup - handles everything in one call.
    
    This function:
    1. Checks Docker is running
    2. Verifies container exists and is healthy
    3. Waits for database to be ready
    4. Fixes listener for ARM Macs
    5. Creates VECTOR user
    6. Tests connection
    """
    print("=" * 60)
    print("🚀 ORACLE DATABASE SETUP")
    print("=" * 60)
    
    # Step 1: Check Docker
    print("\n[1/6] Checking Docker...")
    try:
        result = subprocess.run(['docker', 'info'], capture_output=True, text=True, timeout=10)
        if result.returncode != 0:
            print("   ❌ Docker is not running!")
            print("   💡 Start Docker Desktop and try again.")
            return False
        print("   ✅ Docker is running")
    except FileNotFoundError:
        print("   ❌ Docker not found! Please install Docker.")
        return False
    except subprocess.TimeoutExpired:
        print("   ❌ Docker is not responding. Please restart Docker.")
        return False
    
    # Step 2: Check container
    print(f"\n[2/6] Checking container '{container_name}'...")
    result = subprocess.run(
        ['docker', 'ps', '-a', '--filter', f'name={container_name}', '--format', '{{.Status}}'],
        capture_output=True, text=True
    )
    status = result.stdout.strip()
    
    if not status:
        print(f"   ❌ Container '{container_name}' not found!")
        print("   💡 Run the docker run command from the previous cell first.")
        return False
    elif "Up" not in status:
        print(f"   ⚠️  Container exists but not running. Starting...")
        subprocess.run(['docker', 'start', container_name], capture_output=True)
        time.sleep(5)
    
    print(f"   ✅ Container is running")
    
    def probe_database_ready():
        """True readiness check via SQL: CDB OPEN and FREEPDB1 READ WRITE."""
        probe_sql = """
SET HEADING OFF FEEDBACK OFF PAGESIZE 0 VERIFY OFF ECHO OFF
WHENEVER SQLERROR EXIT SQL.SQLCODE
SELECT status || ':' || open_mode
FROM v$instance
CROSS JOIN (SELECT open_mode FROM v$pdbs WHERE name = 'FREEPDB1');
EXIT;
"""
        probe = subprocess.run(
            ['docker', 'exec', '-i', container_name, 'bash', '-c',
             'export ORACLE_SID=FREE && sqlplus -s / as sysdba'],
            input=probe_sql,
            capture_output=True,
            text=True
        )

        stdout_lines = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
        normalized = " ".join(stdout_lines).upper()
        is_ready = probe.returncode == 0 and "OPEN:READ WRITE" in normalized

        if stdout_lines:
            details = " | ".join(stdout_lines)
        else:
            details = probe.stderr.strip() or f"sqlplus exited with code {probe.returncode}"

        return is_ready, details

    # Step 3: Wait for database ready
    print("\n[3/6] Waiting for database to be ready...")
    print("   (True check: probing instance state and FREEPDB1 open mode)")

    max_wait = 300  # 5 minutes
    check_interval = 5
    elapsed = 0
    last_details = None

    while elapsed < max_wait:
        # Ensure container is still up while waiting
        status_result = subprocess.run(
            ['docker', 'ps', '--filter', f'name={container_name}', '--format', '{{.Status}}'],
            capture_output=True, text=True
        )
        running_status = status_result.stdout.strip()
        if "up" not in running_status.lower():
            print(f"\n   ❌ Container stopped while waiting: {running_status or 'unknown status'}")
            return False

        ready, details = probe_database_ready()
        if ready:
            print("\n   ✅ Database is ready (OPEN:READ WRITE)")
            break

        # Show probe state when it changes
        if details != last_details:
            print(f"\n   🔎 Probe status: {details}")
            last_details = details

        dots = "." * ((elapsed // check_interval) % 4 + 1)
        print(f"\r   ⏳ Waiting{dots.ljust(5)} ({elapsed}s elapsed)", end="", flush=True)
        time.sleep(check_interval)
        elapsed += check_interval
    else:
        print(f"\n   ❌ Timeout waiting for database. Check 'docker exec -it {container_name} bash'")
        return False
    
    # Step 4: Fix listener (for ARM Macs)
    print("\n[4/6] Configuring listener...")
    
    # Fix listener.ora
    subprocess.run(
        ['docker', 'exec', container_name, 'bash', '-c',
         "sed -i 's/HOST = [^)]*)/HOST = 0.0.0.0)/g' /opt/oracle/product/26ai/dbhomeFree/network/admin/listener.ora"],
        capture_output=True
    )
    
    # Restart listener
    subprocess.run(['docker', 'exec', container_name, 'lsnrctl', 'stop'], capture_output=True)
    start_result = subprocess.run(
        ['docker', 'exec', container_name, 'lsnrctl', 'start'],
        capture_output=True, text=True
    )
    
    if "Listening on" not in start_result.stdout:
        print("   ❌ Failed to start listener")
        return False
    
    # Register services
    subprocess.run(
        ['docker', 'exec', container_name, 'bash', '-c',
         "export ORACLE_SID=FREE && sqlplus -s / as sysdba <<< 'ALTER SYSTEM REGISTER;'"],
        capture_output=True
    )
    print("   ✅ Listener configured and running")
    
    # Step 5: Create VECTOR user
    print("\n[5/6] Creating VECTOR user...")
    
    create_user_sql = f'''
    DECLARE
        user_count NUMBER;
    BEGIN
        SELECT COUNT(*) INTO user_count FROM all_users WHERE username = 'VECTOR';
        IF user_count = 0 THEN
            EXECUTE IMMEDIATE 'CREATE USER VECTOR IDENTIFIED BY {vector_password}';
            EXECUTE IMMEDIATE 'GRANT CONNECT, RESOURCE, CREATE SESSION TO VECTOR';
            EXECUTE IMMEDIATE 'GRANT UNLIMITED TABLESPACE TO VECTOR';
            EXECUTE IMMEDIATE 'GRANT CREATE TABLE, CREATE SEQUENCE, CREATE VIEW TO VECTOR';
            DBMS_OUTPUT.PUT_LINE('CREATED');
        ELSE
            DBMS_OUTPUT.PUT_LINE('EXISTS');
        END IF;
    END;
    /
    '''
    
    result = subprocess.run(
        ['docker', 'exec', container_name, 'bash', '-c',
         f"export ORACLE_SID=FREE && sqlplus -s / as sysdba <<< \"ALTER SESSION SET CONTAINER = FREEPDB1; {create_user_sql}\""],
        capture_output=True, text=True
    )
    
    if "ORA-" in result.stdout:
        print(f"   ⚠️  Warning: {result.stdout}")
    else:
        print("   ✅ VECTOR user ready")
    
    # Step 6: Test connection
    print("\n[6/6] Testing connection...")
    try:
        import oracledb
        conn = oracledb.connect(
            user="VECTOR",
            password=vector_password,
            dsn="127.0.0.1:1521/FREEPDB1"
        )
        with conn.cursor() as cur:
            cur.execute("SELECT 1 FROM dual")
            cur.fetchone()
        conn.close()
        print("   ✅ Connection successful!")
    except Exception as e:
        print(f"   ❌ Connection failed: {e}")
        return False
    
    # Success!
    print("\n" + "=" * 60)
    print("🎉 SETUP COMPLETE!")
    print("=" * 60)
    print(f"""
You can now connect to Oracle:
    User: VECTOR
    Password: {vector_password}
    DSN: 127.0.0.1:1521/FREEPDB1
""")
    return True


In [ ]:
# Run this cell after starting your Docker container
# It handles everything: waits for ready, fixes listener, creates user, tests connection
setup_oracle_database()

### Step 3: Connection Helper Function

In the code below we have a reusable function that connects to Oracle Database with automatic retry logic and helpful error messages.

**What it does:**
1. Attempts to connect using the `oracledb` Python driver
2. Retries up to 3 times if the connection fails (useful when the database is still starting)
3. Prints the Oracle version banner on successful connection. This will also include the version you are running
4. Provides troubleshooting hints for common connection errors


In [ ]:
import oracledb
import time

def connect_to_oracle(max_retries=3, retry_delay=5, user="sys", password="OraclePwd_2025", dsn="127.0.0.1:1521/FREEPDB1", program="langchain_oracledb_deep_research_demo"):
    """
    Connect to Oracle database with retry logic and better error handling.
    
    Args:
        max_retries: Maximum number of connection attempts
        retry_delay: Seconds to wait between retries
    """
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user,
                password=password,
                dsn=dsn,
                program=program
            )
            print("✓ Connected successfully!")
            
            # Test the connection
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%';")
                banner = cur.fetchone()[0]
                # Banner should include the version you are running
                print(f"\n{banner}")
            
            return conn
            
        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")
            
            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → This usually means:")
                print("    1. Database is still starting up (wait 2-3 minutes)")
                print("    2. Listener configuration issue")
                print("    3. Container is not running")
                
                if attempt < max_retries:
                    print(f"\n  Waiting {retry_delay} seconds before retry...")
                    time.sleep(retry_delay)
                else:
                    print("\n  💡 Try running: setup_oracle_database()")
                    print("     This will fix the listener and verify the connection.")
                    raise
            else:
                raise
        except Exception as e:
            print(f"✗ Unexpected error: {e}")
            raise
    
    raise ConnectionError("Failed to connect after all retries")

Ensure you have your Docker Engine running before going through the next steps

Connect as the `VECTOR` user dedicated schema for storing embeddings and vector data.


In [ ]:
vector_conn = connect_to_oracle(
    user="VECTOR",
    password="VectorPwd_2025",
    dsn="127.0.0.1:1521/FREEPDB1",
    program="devrel.hub.memory_engineering",
)

print("Using user:", vector_conn.username)

# One-time cleanup: remove prior user-created indexes on the demo vector table.
# This prevents ORA-01408 when rerunning index creation with a new name.
def one_time_cleanup_vector_demo_indexes(conn):
    dropped = []
    with conn.cursor() as cur:
        cur.execute("""
            SELECT index_name
            FROM user_indexes
            WHERE table_name = 'VECTOR_SEARCH_DEMO'
              AND generated = 'N'
        """)
        indexes = [row[0] for row in cur.fetchall()]

        for idx in indexes:
            try:
                cur.execute(f'DROP INDEX "{idx}"')
                dropped.append(idx)
            except Exception as e:
                print(f"  ⚠️ Could not drop index {idx}: {e}")

    conn.commit()
    if dropped:
        print(f"🧹 One-time cleanup: dropped {len(dropped)} old index(es): {', '.join(dropped)}")
    else:
        print("🧹 One-time cleanup: no existing user-created indexes on VECTOR_SEARCH_DEMO")

one_time_cleanup_vector_demo_indexes(vector_conn)


✅ **Setup complete!** You now have Oracle AI Database running locally with an active connection.

Next, we'll create vector-enabled SQL tables using **LangChain's OracleVS integration** to store embeddings and metadata for semantic search.

# Part 2: Vector Search With Langchain and Oracle AI Database

--------

This section demonstrates how to use **LangChain's OracleVS abstraction** over vector-enabled SQL tables to store and search documents using semantic similarity. 

Vector search enables finding documents based on meaning rather than exact keyword matches.

## What You'll Learn

| Step | Description |
|------|-------------|
| **1. Initialize Embeddings** | Load a HuggingFace embedding model to convert text into vectors |
| **2. Create Vector-Enabled Table (OracleVS)** | Set up an Oracle-backed vector-enabled table with cosine distance |
| **3. Create Index** | Build an HNSW (Hierarchical Navigable Small World) index for fast similarity search |
| **4. Add Documents** | Store text with metadata in the vector database |
| **5. Query** | Search for similar documents using natural language |
| **6. Filter Results** | Use metadata filters to narrow down search results |

## Key Components

- **`OracleVS`**: LangChain abstraction over Oracle vector-enabled SQL tables
- **`HuggingFaceEmbeddings`**: Converts text to 768-dimensional vectors
- **`DistanceStrategy.COSINE`**: Measures vector similarity using cosine distance
- **HNSW Index**: Graph-based ANN index for fast and accurate nearest-neighbor retrieval


## Step 1: Creating Vector-Enabled Tables with LangChain OracleVS

In [ ]:
from langchain_oracledb.vectorstores import OracleVS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores.oraclevs import create_index
from langchain_community.vectorstores.utils import DistanceStrategy

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

# Initialize the OracleVS handle over a vector-enabled SQL table
vector_store = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name="VECTOR_SEARCH_DEMO",
    distance_strategy=DistanceStrategy.COSINE,
)


In [ ]:
# Helper to safely create index (skips if already exists)
def safe_create_index(conn, vs, idx_name):
    """Create index, skipping if it already exists."""
    try:
        create_index(
            client=conn,
            vector_store=vs,
            params={"idx_name": idx_name, "idx_type": "HNSW"}
        )
        print(f"  ✅ Created index: {idx_name}")
    except Exception as e:
        if "ORA-00955" in str(e):
            print(f"  ⏭️ Index already exists: {idx_name} (skipped)")
        else:
            raise


In [ ]:
import logging

# Suppress langchain_oracledb logging, remove this if you want to see the debug logs
logging.getLogger("langchain_oracledb").setLevel(logging.CRITICAL)

# Create an HNSW index for fast similarity search
safe_create_index(vector_conn, vector_store, "oravs_hnsw")


## Step 2: Ingesting Research Paper Data


In [ ]:
from datasets import load_dataset

MAX_PAPERS = 1000
ds_stream = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

sampled_papers = []
texts = []
metadata = []

for i, item in enumerate(ds_stream):
    if i >= MAX_PAPERS:
        break

    arxiv_id = item.get("arxiv_id", f"unknown_{i}")
    title = (item.get("title") or "").strip()
    abstract = (item.get("abstract") or "").strip()
    primary_subject = (item.get("primary_subject") or "").strip()
    authors = item.get("authors") or []

    if isinstance(authors, str):
        authors_text = authors
    elif isinstance(authors, list):
        authors_text = ", ".join(str(a).strip() for a in authors if str(a).strip())
    else:
        authors_text = ""

    text = f"Title: {title}\nAbstract: {abstract}"

    sampled_papers.append({
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "primary_subject": primary_subject,
        "authors": authors_text,
    })
    texts.append(text)
    metadata.append({
        "id": arxiv_id,
        "arxiv_id": arxiv_id,
        "title": title,
        "primary_subject": primary_subject,
        "authors": authors_text,
    })

vector_store.add_texts(
    texts=texts,
    metadatas=metadata,
)

print(f"✅ Ingested {len(texts)} research papers into VECTOR_SEARCH_DEMO")


In [ ]:
sample_primary_subject = sampled_papers[0]["primary_subject"] if sampled_papers else ""
sample_arxiv_id = sampled_papers[0]["arxiv_id"] if sampled_papers else ""
print("Sample primary subject:", sample_primary_subject)
print("Sample arxiv_id:", sample_arxiv_id)


## Step 3: Querying Vector-Enabled SQL Tables

Search for documents similar to a natural language query. 

The OracleVS layer converts queries to embeddings and finds the closest matches.


Basic Search

In [ ]:
query = "Find research papers about planetary exploration mission planning."

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)


Search With Scores

In [ ]:
results = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results:
    print("Score:", score)
    print("Text :", doc.page_content)
    print("Meta :", doc.metadata)
    print("------")

Filter by exact match on a metadata field

In [ ]:
query = "Find papers related to mission planning and observational astronomy."

# This returns docs where metadata.primary_subject matches the sampled subject.
docs = vector_store.similarity_search(
    query,
    k=3,
    filter={"primary_subject": {"$eq": sample_primary_subject}},
)

for doc in docs:
    print("Text:", doc.page_content[:120], "...")
    print("Meta:", doc.metadata)
    print("------")


Filter by id list ($in)

In [ ]:
docs = vector_store.similarity_search(
    query="Explain key themes in this research paper",
    k=5,
    filter={"id": {"$in": [sample_arxiv_id]}},
)

print(docs)


# Part 3: Memory Engineering and Agent Memory
--------



**`Agent Memory`** is the exocortex that augments an LLM—capturing, encoding, storing, linking, and retrieving information beyond the model's parametric and contextual limits. 
It provides the persistence and structure required for long-horizon reasoning and reliable behaviour.

**`Memory Engineering`** is the scaffolding and control harness that we design to move information optimally and efficiently into, through, and across all components of an AI system(databases, LLMs, applications etc). It ensures that data is captured, transformed, organized, and retrieved in the right way at the right time—so agents can behave reliably, believably, and capabaly.

This is the core section of the notebook where we build a complete **`Memory Manager`** for AI agents. 

Just like humans have different types of memory (short-term, long-term, procedural), AI agents benefit from specialized memory systems.

## Why Memory Engineering Matters

Without memory, agents:
- Forget previous conversations
- Can't learn from past interactions
- Repeat the same mistakes
- Lack context for complex tasks

With proper memory engineering, agents can:
- Maintain context across sessions
- Learn and improve over time
- Access relevant knowledge when needed
- Execute complex multi-step workflows

## Memory Types We'll Implement

| Memory Type | Human Analogy | Purpose | Storage |
|-------------|---------------|---------|---------|
| **Conversational** | Short-term memory | Chat history per thread | SQL Table |
| **Knowledge Base** | Long-term semantic memory | Facts, documents, search results | Vector-Enabled SQL Table |
| **Workflow** | Procedural memory | Learned action patterns | Vector-Enabled SQL Table |
| **Toolbox** | Skill memory | Available tools & capabilities | Vector-Enabled SQL Table |
| **Entity** | Episodic memory | People, places, systems mentioned | Vector-Enabled SQL Table |
| **Summary** | Compressed memory | Condensed context for long conversations | Vector-Enabled SQL Table |
| **Tool Log** | Episodic memory | Raw tool call outputs offloaded from context | SQL Table |

> **Note on Tool Log:** Tool Log is a form of episodic memory — it records *what happened* during each tool execution. Beyond keeping the context window lean, tool logs can serve as a source from which **procedural memories** (workflow patterns) and **semantic memories** (knowledge base entries) can be distilled over time.

## Steps in This Section

1. **Define table names** for each memory type
2. **Create SQL table** for conversational history
3. **Create SQL table** for tool call logs (experimental memory)
4. **Create vector-enabled SQL tables** for semantic memories
5. **Build indexes** for fast similarity search
6. **Implement MemoryLayer class** with read/write methods for each memory type
7. **Initialize the memory manager** with all storage backends

## Step 1: Define Memory Tables and Stores
First, we define table names for each memory type. 

These tables will be created in Oracle Database to persist agent memory.

In [ ]:
# Table names for each memory type
CONVERSATIONAL_TABLE   = "CONVERSATIONAL_MEMORY" # Episodic memory
KNOWLEDGE_BASE_TABLE   = "SEMANTIC_MEMORY" # Semantic memory
WORKFLOW_TABLE = "WORKFLOW_MEMORY" # Procedural memory
TOOLBOX_TABLE    = "TOOLBOX_MEMORY" # Procedural memory
ENTITY_TABLE = "ENTITY_MEMORY" # Semantic memory
SUMMARY_TABLE = "SUMMARY_MEMORY" # Semanatic memory
TOOL_LOG_TABLE = "TOOL_LOG" # Episodic

ALL_TABLES = [CONVERSATIONAL_TABLE, KNOWLEDGE_BASE_TABLE, WORKFLOW_TABLE, TOOLBOX_TABLE, ENTITY_TABLE, SUMMARY_TABLE, TOOL_LOG_TABLE]

# Drop existing tables to start fresh
for table in ALL_TABLES:
    try:
        with vector_conn.cursor() as cur:
            cur.execute(f"DROP TABLE {table} PURGE")
    except Exception as e:
        if "ORA-00942" in str(e):
            print(f"  - {table} (not exists)")
        else:
            print(f"  ✗ {table}: {e}")
            
vector_conn.commit()

In [ ]:
# Model token limits (for context management)
MODEL_TOKEN_LIMITS = {
    "gpt-5": 256000,
    "gpt-5-mini": 128000,
    "gpt-4o": 128000,
    "gpt-5": 128000,
    "gpt-4-turbo": 128000,
    "gpt-4": 8192,
    "gpt-3.5-turbo": 16385,
}

### Step 1a: Create Conversational Memory Table

This function below creates a SQL table to store chat history. 

Unlike semantic memories backed by vector-enabled SQL tables, conversational memory uses a traditional table because we need exact retrieval by thread ID (not similarity search).

**What it does:**
- Creates a table with columns: `id`, `thread_id`, `role`, `content`, `timestamp`, `metadata`
- Adds an index on `thread_id` for fast conversation lookups
- Adds an index on `timestamp` for chronological ordering


In [ ]:
def create_conversational_history_table(conn, table_name: str = "CONVERSATIONAL_MEMORY"):
    """
    Create a table to store conversational history.

    Args:
        conn: Oracle database connection
        table_name: Name of the table to create
    """
    with conn.cursor() as cur:
        # Drop table if exists
        try:
            cur.execute(f"DROP TABLE {table_name}")
        except:
            pass  # Table doesn't exist
        
        # Create table with proper schema
        cur.execute(f"""
            CREATE TABLE {table_name} (
                id VARCHAR2(100) DEFAULT SYS_GUID() PRIMARY KEY,
                thread_id VARCHAR2(100) NOT NULL,
                role VARCHAR2(50) NOT NULL,
                content CLOB NOT NULL,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                metadata CLOB,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                summary_id VARCHAR2(100) DEFAULT NULL
            )
        """)
        
        # Create index on thread_id for faster lookups
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_thread_id ON {table_name}(thread_id)
        """)
        
        # Create index on timestamp for ordering
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_timestamp ON {table_name}(timestamp)
        """)
        
    conn.commit()
    print(f"Table {table_name} created successfully with indexes")
    return table_name


In [ ]:
# Create the table
CONVERSATION_HISTORY_TABLE = create_conversational_history_table(vector_conn, CONVERSATIONAL_TABLE)

### Step 1b: Create Tool Log Table (Experimental Memory)

Tool call outputs during agent execution can **bloat the context window** quickly — a single web search might return thousands of tokens that are only needed once. 

The `TOOL_LOG` table acts as an **experimental memory**: full tool outputs are persisted to the database and replaced in the context window with a compact one-line reference. The agent can retrieve full outputs later if needed via `read_tool_log`.

This is a form of **context offloading** — keeping the working memory lean while preserving full fidelity in durable storage.

In [ ]:
def create_tool_log_table(conn, table_name: str = "TOOL_LOG"):
    """Create a table to log tool call outputs (experimental memory)."""
    with conn.cursor() as cur:
        try:
            cur.execute(f"DROP TABLE {table_name}")
        except:
            pass

        cur.execute(f"""
            CREATE TABLE {table_name} (
                id VARCHAR2(100) DEFAULT SYS_GUID() PRIMARY KEY,
                thread_id VARCHAR2(100) NOT NULL,
                tool_call_id VARCHAR2(200) NOT NULL,
                tool_name VARCHAR2(200) NOT NULL,
                tool_args CLOB,
                tool_output CLOB,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        cur.execute(f"CREATE INDEX idx_{table_name.lower()}_thread ON {table_name}(thread_id)")
    conn.commit()
    print(f"Table {table_name} created successfully")
    return table_name

TOOL_LOG_TABLE_NAME = create_tool_log_table(vector_conn, TOOL_LOG_TABLE)

### Step 1c: Create Vector-Enabled Tables for Each Memory Type

Here we create 5 separate OracleVS-backed vector-enabled SQL tables—one for each memory type. 

Each semantic memory is backed by its own Oracle table with a VECTOR column and uses the same embedding model for consistency.

| Vector-Enabled Table Handle | Purpose |
|--------------|---------|
| `knowledge_base_vs` | Store documents, facts, and search results |
| `workflow_vs` | Store learned action patterns and tool sequences |
| `toolbox_vs` | Store tool definitions for semantic tool discovery |
| `entity_vs` | Store extracted entities (people, places, systems) |
| `summary_vs` | Store compressed summaries for long conversations |


In [ ]:
knowledge_base_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=KNOWLEDGE_BASE_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

workflow_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=WORKFLOW_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

toolbox_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=TOOLBOX_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

entity_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=ENTITY_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

summary_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=SUMMARY_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)


Then we create indexes for each vector-enabled table

In [ ]:
print("Creating vector indexes...")
safe_create_index(vector_conn, knowledge_base_vs, "knowledge_base_vs_hnsw")
safe_create_index(vector_conn, workflow_vs, "workflow_vs_hnsw")
safe_create_index(vector_conn, toolbox_vs, "toolbox_vs_hnsw")
safe_create_index(vector_conn, entity_vs, "entity_vs_hnsw")
safe_create_index(vector_conn, summary_vs, "summary_vs_hnsw")
print("All indexes created!")

if "sampled_papers" in globals() and sampled_papers:
    kb_texts = [f"Title: {p['title']}\nAbstract: {p['abstract']}" for p in sampled_papers]
    kb_meta = [
        {
            "id": p["arxiv_id"],
            "arxiv_id": p["arxiv_id"],
            "title": p["title"],
            "primary_subject": p["primary_subject"],
            "authors": p["authors"],
            "source_type": "arxiv_papers",
        }
        for p in sampled_papers
    ]
    knowledge_base_vs.add_texts(kb_texts, kb_meta)
    print(f"✅ Seeded knowledge base memory with {len(kb_texts)} arXiv papers")


## Step 2: Programmatic vs Agent-Triggered Operations


A key design decision in memory engineering is deciding which operations run **programmatically** (always executed by the harness) versus **agent-triggered** (the LLM chooses to invoke them during the loop).

In this notebook, the harness is intentionally opinionated: memory loading and persistence are automatic, while external/expansion actions are chosen by the agent.

| Operation | Programmatic | Agent-Triggered | Notes |
|-----------|:------------:|:---------------:|-------|
| `read_conversational_memory()` | ✅ | ❌ | Always loaded at loop start (unsummarized units only) |
| `read_knowledge_base()` | ✅ | ❌ | Always loaded at loop start |
| `read_workflow()` | ✅ | ❌ | Always loaded at loop start |
| `read_entity()` | ✅ | ❌ | Always loaded at loop start |
| `read_summary_context()` | ✅ | ❌ | Always loaded at loop start (IDs + descriptions) |
| `read_toolbox()` | ✅ | ❌ | Tool schemas are retrieved before model reasoning |
| `write_conversational_memory()` | ✅ | ❌ | User message (pre-loop) + assistant answer (post-loop) |
| `write_workflow()` | ✅ | ❌ | Persisted after loop when tool steps exist |
| `write_entity()` | ✅ | ❌ | Best-effort extraction around user/final assistant text |
| `write_tool_log()` | ✅ | ❌ | Full tool output offloaded to DB after every tool execution |
| Tool-call decision (`tool_choice=auto`) | ❌ | ✅ | Model decides whether to call tools |
| `search_tavily()` | ❌ | ✅ | Agent-triggered external retrieval |
| `expand_summary()` | ❌ | ✅ | Agent-triggered just-in-time summary expansion |
| `summarize_and_store()` | ❌ | ✅ | Agent-triggered context compaction primitive |
| `summarize_conversation()` | ❌ | ✅ | Agent-triggered conversation compaction for active thread |

### What Is Programmatic in This Harness

These operations are always executed by code, not delegated to the model:

1. **Context assembly** at the start of `call_agent()`.
2. **Tool schema retrieval** before each model call.
3. **Memory persistence** around the loop (store user turn, store assistant turn, persist workflow/entity updates).
4. **Tool execution dispatch** after a tool call is chosen (once selected by the model, execution is deterministic in code).
5. **Tool output offloading** via `write_tool_log()` — full outputs are persisted to the database and replaced with compact references in the context window.

### What Is Agent-Triggered in This Harness

These operations are chosen by the model during the loop:

1. **Whether** to call a tool at all.
2. **Which** tool to call.
3. **When** to trigger web search, summary expansion, or conversation compaction.
4. **How** to sequence multiple tool calls before finalizing an answer.

### Why This Split Works for Memory-Centric Agents

1. **Reliability from programmatic memory** — critical memory load/save behavior never depends on the model remembering to do it.
2. **Adaptivity from agent-triggered tools** — the model can selectively fetch/expand/compact only when needed.
3. **Clear control boundaries** — the harness owns state integrity; the model owns strategy inside those boundaries.

## Step 3: Memory Manager Implementation

The `MemoryManager` class is the central abstraction that unifies all memory operations. It provides a clean interface for reading and writing to different memory types, hiding the complexity of SQL queries and vector-enabled table operations.

### What We're Building

A single class that manages 6 types of memory with consistent read/write patterns:

| Memory Type | Storage | Write Method | Read Method |
|-------------|---------|--------------|-------------|
| **Conversational** | SQL Table | `write_conversational_memory()` | `read_conversational_memory()` |
| **Knowledge Base** | Vector-Enabled SQL Table | `write_knowledge_base()` | `read_knowledge_base()` |
| **Workflow** | Vector-Enabled SQL Table | `write_workflow()` | `read_workflow()` |
| **Toolbox** | Vector-Enabled SQL Table | `write_toolbox()` | `read_toolbox()` |
| **Entity** | Vector-Enabled SQL Table | `write_entity()` | `read_entity()` |
| **Summary** | Vector-Enabled SQL Table | `write_summary()` | `read_summary_memory()`, `read_summary_context()` |

### Key Features

- **Thread-based conversations** — Messages are organized by `thread_id` for multi-conversation support
- **Semantic search** — Vector-enabled SQL tables enable finding relevant content by meaning, not just keywords
- **Metadata filtering** — Workflows filter by `num_steps > 0`, summaries filter by `id`
- **LLM-powered entity extraction** — Automatically extracts people, places, and systems from text
- **Formatted context output** — Each read method returns formatted text ready for the LLM context

### Alternative: Memory Manager Frameworks

There are existing frameworks that abstract memory management for AI agents:

| Framework | Description |
|-----------|-------------|
| **LangChain Memory** | Built-in memory classes (ConversationBufferMemory, VectorStoreRetrieverMemory) |
| **Mem0** | Dedicated memory layer for AI agents with automatic memory management |
| **LlamaIndex** | Document-based memory with various storage backends |
| **Zep** | Long-term memory service for AI assistants |

### Pros and Cons of Building Your Own

| Approach | Pros | Cons |
|----------|------|------|
| **Custom (what we're doing)** | Full control, tailored to your needs, deeper understanding, no external dependencies | More code to maintain, need to handle edge cases yourself |
| **Using a framework** | Faster to implement, battle-tested, community support, handles edge cases | Less control, may not fit your exact use case, additional dependency |

> **For learning purposes**, building your own memory manager (as we do here) gives you a deep understanding of how memory engineering works. 
> 
> **For production**, you might consider using or extending an existing framework. 
>
> For example, this simple notebook only illustrates reads and writes, but not deletion and updates.

In [ ]:
import json as json_lib
from datetime import datetime

class MemoryManager:
    """
    A simplified memory manager for AI agents using Oracle AI Database.
    
    Manages 6 types of memory:
    - Conversational: Chat history per thread (SQL table)
    - Knowledge Base: Searchable documents (vector-enabled SQL table)
    - Workflow: Execution patterns (vector-enabled SQL table)
    - Toolbox: Available tools (vector-enabled SQL table)
    - Entity: People, places, systems (vector-enabled SQL table)
    - Summary: Storing compressed context window
    - Tool Log: Experimental memory that offloads tool outputs from context window (SQL table)
    """
    
    def __init__(self, conn, conversation_table: str, knowledge_base_vs, workflow_vs, toolbox_vs, entity_vs, summary_vs, tool_log_table: str = None):
        self.conn = conn
        self.conversation_table = conversation_table
        self.knowledge_base_vs = knowledge_base_vs
        self.workflow_vs = workflow_vs
        self.toolbox_vs = toolbox_vs
        self.entity_vs = entity_vs
        self.summary_vs = summary_vs
        self.tool_log_table = tool_log_table
    
    # ==================== CONVERSATIONAL MEMORY (SQL) ====================
    
    def write_conversational_memory(self, content: str, role: str, thread_id: str) -> str:
        """Store a message in conversation history."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            id_var = cur.var(str)
            cur.execute(f"""
                INSERT INTO {self.conversation_table} (thread_id, role, content, metadata, timestamp)
                VALUES (:thread_id, :role, :content, :metadata, CURRENT_TIMESTAMP)
                RETURNING id INTO :id
            """, {"thread_id": thread_id, "role": role, "content": content, "metadata": "{}", "id": id_var})
            record_id = id_var.getvalue()[0] if id_var.getvalue() else None
        self.conn.commit()
        return record_id
    
    def get_unsummarized_messages(self, thread_id: str, limit: int = 100) -> list[dict]:
        """Return unsummarized conversation units for a thread."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            cur.execute(f"""
                SELECT id, role, content, timestamp
                FROM {self.conversation_table}
                WHERE thread_id = :thread_id AND summary_id IS NULL
                ORDER BY timestamp ASC
                FETCH FIRST :limit ROWS ONLY
            """, {"thread_id": thread_id, "limit": limit})
            rows = cur.fetchall()

        return [
            {"id": rid, "role": role, "content": content, "timestamp": ts}
            for rid, role, content, ts in rows
        ]

    def read_conversational_memory(self, thread_id: str, limit: int = 10) -> str:
        """Read unsummarized conversation history for a thread.
        
        NOTE: Only returns messages where summary_id IS NULL. Once messages are
        summarized via summarize_conversation(), they are excluded here and
        replaced by a compact summary reference in Summary Memory.
        """
        messages = self.get_unsummarized_messages(thread_id, limit=limit)
        lines = [f"[{m['timestamp'].strftime('%H:%M:%S')}] [{m['role']}] {m['content']}" for m in messages]
        messages_formatted = '\n'.join(lines)
        return f"""## Conversation Memory (Thread: {thread_id})
### Purpose: Recent dialogue turns that have NOT yet been summarized.
### When to use: Refer to this for the user's latest questions, your prior answers, and any
### commitments or follow-ups from the current conversation. If context grows too long,
### call summarize_conversation(thread_id) to compact older turns into Summary Memory.

{messages_formatted}"""

    def mark_as_summarized(self, thread_id: str, summary_id: str, message_ids: list[str] | None = None):
        """Mark conversation units as summarized."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            if message_ids:
                cur.executemany(
                    f"""
                    UPDATE {self.conversation_table}
                    SET summary_id = :summary_id
                    WHERE thread_id = :thread_id AND id = :id AND summary_id IS NULL
                    """,
                    [{"summary_id": summary_id, "thread_id": thread_id, "id": mid} for mid in message_ids],
                )
                count = len(message_ids)
            else:
                cur.execute(f"""
                    UPDATE {self.conversation_table}
                    SET summary_id = :summary_id
                    WHERE thread_id = :thread_id AND summary_id IS NULL
                """, {"summary_id": summary_id, "thread_id": thread_id})
                count = cur.rowcount
        self.conn.commit()
        print(f"  📦 Marked {count} messages as summarized (summary_id: {summary_id})")

    # ==================== KNOWLEDGE BASE (Vector-Enabled SQL Table) ====================
    
    def write_knowledge_base(self, text: str, metadata: dict):
        """Store text in knowledge base with metadata."""
        self.knowledge_base_vs.add_texts([text], [metadata])
    
    def read_knowledge_base(self, query: str, k: int = 3) -> str:
        """Search knowledge base for relevant content."""
        results = self.knowledge_base_vs.similarity_search(query, k=k)
        content = "\n".join([doc.page_content for doc in results])
        return f"""## Knowledge Base Memory
### Purpose: Factual documents, research papers, and web search results stored for long-term reference.
### When to use: Treat this as your primary source of evidence. Cite specific facts, titles, or
### findings from here before resorting to external search. If the knowledge base lacks what you
### need, use search_tavily() to fetch new information (which will be stored here automatically).

{content}"""
    
    
    # ==================== WORKFLOW (Vector-Enabled SQL Table) ====================
    
    def write_workflow(self, query: str, steps: list, final_answer: str, success: bool = True):
        """Store a completed workflow pattern for future reference."""
        # Format steps as text
        steps_text = "\n".join([f"Step {i+1}: {s}" for i, s in enumerate(steps)])
        text = f"Query: {query}\nSteps:\n{steps_text}\nAnswer: {final_answer[:200]}"
        
        metadata = {
            "query": query,
            "success": success,
            "num_steps": len(steps),
            "timestamp": datetime.now().isoformat()
        }
        self.workflow_vs.add_texts([text], [metadata])
    
    def read_workflow(self, query: str, k: int = 3) -> str:
        """Search for similar past workflows with at least 1 step."""
        # Filter to only include workflows that have steps (num_steps > 0)
        results = self.workflow_vs.similarity_search(
            query, 
            k=k, 
            filter={"num_steps": {"$gt": 0}}
        )
        if not results:
            return "## Workflow Memory\nNo relevant workflows found."
        content = "\n---\n".join([doc.page_content for doc in results])
        return f"""## Workflow Memory
### Purpose: Step-by-step records of how similar past queries were resolved (tool calls and outcomes).
### When to use: Before planning a multi-step action, check if a similar workflow already succeeded.
### Reuse proven tool sequences instead of re-discovering them. Skip workflows marked as failed.

{content}"""
    
    # ==================== TOOLBOX (Vector-Enabled SQL Table) ====================
    
    def write_toolbox(self, text: str, metadata: dict):
        """Store a tool definition in the toolbox."""
        self.toolbox_vs.add_texts([text], [metadata])
    
    def read_toolbox(self, query: str, k: int = 3) -> list[dict]:
        """Find relevant tools and return OpenAI-compatible schemas."""
        results = self.toolbox_vs.similarity_search(query, k=k)
        tools = []
        for doc in results:
            meta = doc.metadata
            # Extract parameters from metadata and convert to OpenAI format
            stored_params = meta.get("parameters", {})
            properties = {}
            required = []
            
            for param_name, param_info in stored_params.items():
                # Convert stored param info to OpenAI schema format
                param_type = param_info.get("type", "string")
                # Map Python types to JSON schema types
                type_mapping = {
                    "<class 'str'>": "string",
                    "<class 'int'>": "integer", 
                    "<class 'float'>": "number",
                    "<class 'bool'>": "boolean",
                    "str": "string",
                    "int": "integer",
                    "float": "number",
                    "bool": "boolean"
                }
                json_type = type_mapping.get(param_type, "string")
                properties[param_name] = {"type": json_type}
                
                # If no default, it's required
                if "default" not in param_info:
                    required.append(param_name)
            
            tools.append({
                "type": "function",
                "function": {
                    "name": meta.get("name", "tool"),
                    "description": meta.get("description", ""),
                    "parameters": {"type": "object", "properties": properties, "required": required}
                }
            })
        return tools

    # ==================== ENTITY (Vector-Enabled SQL Table) ====================
    
    def extract_entities(self, text: str, llm_client) -> list[dict]:
        """Use LLM to extract entities (people, places, systems) from text."""
        if not text or len(text.strip()) < 5:
            return []
        
        prompt = f'''Extract entities from: "{text[:500]}"
Return JSON: [{{"name": "X", "type": "PERSON|PLACE|SYSTEM", "description": "brief"}}]
If none: []'''

        try:
            response = llm_client.chat.completions.create(
                model="gpt-5",
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=300
            )
            result = response.choices[0].message.content.strip()
            
            # Extract JSON array from response
            start, end = result.find("["), result.rfind("]")
            if start == -1 or end == -1:
                return []
            
            parsed = json_lib.loads(result[start:end+1])
            return [{"name": e["name"], "type": e.get("type", "UNKNOWN"), "description": e.get("description", "")} 
                    for e in parsed if isinstance(e, dict) and e.get("name")]
        except:
            return []
    
    def write_entity(self, name: str, entity_type: str, description: str, llm_client=None, text: str = None):
        """Store an entity OR extract and store entities from text."""
        if text and llm_client:
            # Extract and store entities from text
            entities = self.extract_entities(text, llm_client)
            for e in entities:
                self.entity_vs.add_texts(
                    [f"{e['name']} ({e['type']}): {e['description']}"],
                    [{"name": e['name'], "type": e['type'], "description": e['description']}]
                )
            return entities
        else:
            # Store single entity directly
            self.entity_vs.add_texts(
                [f"{name} ({entity_type}): {description}"],
                [{"name": name, "type": entity_type, "description": description}]
            )
    
    def read_entity(self, query: str, k: int = 5) -> str:
        """Search for relevant entities."""
        results = self.entity_vs.similarity_search(query, k=k)
        if not results:
            return "## Entity Memory\nNo entities found."
        
        entities = [f"• {doc.metadata.get('name', '?')}: {doc.metadata.get('description', '')}" 
                    for doc in results if hasattr(doc, 'metadata')]
        entities_formatted = '\n'.join(entities)
        return f"""## Entity Memory
### Purpose: Named entities (people, places, systems, paper titles) extracted from conversations.
### When to use: Use these to resolve references like "that author" or "the system we discussed".
### Entity memory provides continuity across turns — ground your answers in known entities
### rather than guessing or re-asking the user for names and details already mentioned.

{entities_formatted}"""
    
    # ==================== SUMMARY (Vector-Enabled SQL Table) ====================
    
    def write_summary(self, summary_id: str, full_content: str, summary: str, description: str):
        """Store a summary with its original content."""
        self.summary_vs.add_texts(
            [f"{summary_id}: {description}"],
            [{"id": summary_id, "full_content": full_content, "summary": summary, "description": description}]
        )
        return summary_id
    
    def read_summary_memory(self, summary_id: str) -> str:
        """Retrieve a specific summary by ID (just-in-time retrieval)."""
        results = self.summary_vs.similarity_search(
            summary_id, 
            k=5, 
            filter={"id": summary_id}
        )
        if not results:
            return f"Summary {summary_id} not found."
        doc = results[0]
        return doc.metadata.get('summary', 'No summary content.')
    
    def read_summary_context(self, query: str = "", k: int = 10) -> str:
        """Get available summaries for context window (IDs + descriptions only)."""
        results = self.summary_vs.similarity_search(query or "summary", k=k)
        if not results:
            return "## Summary Memory\nNo summaries available."
        
        lines = [
            "## Summary Memory",
            "### Purpose: Compressed snapshots of older conversations and context windows.",
            "### When to use: These are lightweight pointers. If a summary looks relevant,",
            "### call expand_summary(summary_id) to retrieve the full content just-in-time.",
            "### Do NOT expand all summaries — only expand when you need specific details.",
            ""
        ]
        for doc in results:
            sid = doc.metadata.get('id', '?')
            desc = doc.metadata.get('description', 'No description')
            lines.append(f"  • [ID: {sid}] {desc}")
        return "\n".join(lines)
    
    # ==================== TOOL LOG (SQL - Experimental Memory) ====================
    
    def write_tool_log(self, thread_id: str, tool_call_id: str, tool_name: str, tool_args: str, tool_output: str) -> str:
        """Log a tool call output to the database and return a compact reference."""
        if not self.tool_log_table:
            return tool_output  # Fallback: return full output if no table configured
        
        with self.conn.cursor() as cur:
            id_var = cur.var(str)
            cur.execute(f"""
                INSERT INTO {self.tool_log_table} (thread_id, tool_call_id, tool_name, tool_args, tool_output)
                VALUES (:thread_id, :tool_call_id, :tool_name, :tool_args, :tool_output)
                RETURNING id INTO :id
            """, {
                "thread_id": str(thread_id), "tool_call_id": tool_call_id,
                "tool_name": tool_name, "tool_args": tool_args,
                "tool_output": tool_output, "id": id_var
            })
            log_id = id_var.getvalue()[0] if id_var.getvalue() else None
        self.conn.commit()
        
        # Return a compact reference instead of the full output
        preview = tool_output[:150].replace("\n", " ")
        return f"[Tool Log {log_id}] {tool_name} executed. Preview: {preview}..."
    
    def read_tool_log(self, thread_id: str, limit: int = 20) -> list[dict]:
        """Read tool call logs for a thread."""
        if not self.tool_log_table:
            return []
        with self.conn.cursor() as cur:
            cur.execute(f"""
                SELECT id, tool_call_id, tool_name, tool_args, tool_output, timestamp
                FROM {self.tool_log_table}
                WHERE thread_id = :thread_id
                ORDER BY timestamp DESC
                FETCH FIRST :limit ROWS ONLY
            """, {"thread_id": str(thread_id), "limit": limit})
            rows = cur.fetchall()
        return [
            {"id": r[0], "tool_call_id": r[1], "tool_name": r[2],
             "tool_args": r[3], "tool_output": r[4], "timestamp": r[5]}
            for r in rows
        ]

    # ==================== SUMMARY EXPANSION HELPERS ====================

    def get_messages_by_summary_id(self, summary_id: str) -> list[dict]:
        """Retrieve original conversation messages that were compacted into a given summary."""
        with self.conn.cursor() as cur:
            cur.execute(f"""
                SELECT id, role, content, timestamp
                FROM {self.conversation_table}
                WHERE summary_id = :summary_id
                ORDER BY timestamp ASC
            """, {"summary_id": summary_id})
            rows = cur.fetchall()
        return [
            {"id": rid, "role": role, "content": content, "timestamp": ts}
            for rid, role, content, ts in rows
        ]

In [ ]:
# Initialize the MemoryLayer instance
# Note: Uses SQL table for conversational memory, vector-enabled SQL tables for others
memory_manager = MemoryManager(
    conn=vector_conn,
    conversation_table=CONVERSATION_HISTORY_TABLE, 
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs,
    tool_log_table=TOOL_LOG_TABLE_NAME  # Experimental memory: offloads tool outputs from context
)

## Step 4: Creating the Agent's Toolbox

### The Scalability Problem with Tools

As your AI system grows, you might have **hundreds of tools** available—APIs, database queries, calculators, search engines, and more. However, passing all tools to the LLM at inference time creates serious problems:

| Problem | Impact |
|---------|--------|
| **Context bloat** | Tool definitions consume tokens, leaving less room for actual content |
| **Tool selection failure** | LLMs struggle to choose the right tool when presented with too many options |
| **Increased latency** | More tokens = slower inference |
| **Higher costs** | More tokens = higher API costs |

Model providers like OpenAI and Anthropic typically recommend limiting the number of tools exposed to an LLM (often 10-20 max for reliable selection).

### The Solution: Semantic Tool Retrieval

The `Toolbox` class solves this by treating tools as a **searchable memory**:

1. **Register hundreds of tools** — Store all available tools with their descriptions and embeddings
2. **Retrieve only relevant tools** — At inference time, use vector search to find tools semantically relevant to the current query
3. **Pass a focused toolset** — Only the retrieved tools (typically 3-5) are passed to the LLM

This approach means your system can **scale to hundreds of tools** while the LLM only sees the most relevant ones for each query.

### How the Code Works

The `Toolbox` class uses **docstrings as the retrieval key**:

```
User Query → Embed Query → Vector Search → Find tools with similar docstrings → Return relevant tools
```

| Component | Purpose |
|-----------|---------|
| `get_embedding()` | Converts tool description to a vector |
| `ToolMetadata` | Pydantic model storing tool name, description, signature, parameters |
| `_augment_docstring()` | Uses LLM to improve the docstring for better retrieval |
| `_generate_queries()` | Creates synthetic queries that would trigger this tool |
| `register_tool()` | Decorator that stores tool with its embedding in the toolbox |

When you call `memory_manager.read_toolbox(query)`, it performs a similarity search to find tools whose docstrings are semantically similar to the query.

### The Intersection of Three Engineering Disciplines

This implementation combines techniques from **memory engineering**, **context engineering**, and **prompt engineering**:

| Discipline | Technique Used | How It Helps |
|------------|----------------|--------------|
| **Memory Engineering** | Toolbox as procedural memory | Tools are stored and retrieved like learned skills |
| **Memory Engineering** | Docstring augmentation | LLM improves docstrings for better semantic retrieval |
| **Memory Engineering** | Synthetic query generation | Creates example queries to improve tool discoverability |
| **Context Engineering** | Selective tool retrieval | Only relevant tools enter the context, reducing bloat |
| **Context Engineering** | Context offloading | Tool results can be summarized to save context space |
| **Prompt Engineering** | Role setting | "You are a technical writer" improves docstring quality |

### Key Insight

The `augment=True` flag in `@toolbox.register_tool(augment=True)` triggers:
1. **Docstring augmentation** — LLM rewrites the docstring to be clearer and more searchable
2. **Synthetic query generation** — LLM generates example queries that would need this tool
3. **Rich embedding** — Combines name + augmented docstring + signature + queries for better retrieval

This means a simple one-line docstring like `"Search the web"` becomes a rich, detailed description that's much more likely to be retrieved when the user asks something like `"What's the latest news about AI?"`

In [ ]:
import inspect
import uuid
from typing import Callable, Optional, Union
from pydantic import BaseModel

def get_embedding(text: str) -> list[float]:
    """
    Get the embedding for a text using the configured embedding model.
    """
    return embedding_model.embed_query(text)


class ToolMetadata(BaseModel):
    """Metadata for a registered tool."""
    name: str
    description: str
    signature: str
    parameters: dict
    return_type: str


class Toolbox:
    """
    A toolbox for registering, storing, and retrieving tools with LLM-powered augmentation.
    
    Tools are stored with embeddings for semantic retrieval, allowing the agent to
    find relevant tools based on natural language queries.
    """
    
    def __init__(self, memory_manager, llm_client, model: str = "gpt-5"):
        """
        Initialize the Toolbox.
        
        Args:
            memory_manager: MemoryManager instance for storing tools
            llm_client: OpenAI client for LLM augmentation
            model: Model to use for augmentation (default: gpt-5)
        """
        self.memory_manager = memory_manager
        self.llm_client = llm_client
        self.model = model
        self._tools: dict[str, Callable] = {}  # Maps tool_id -> callable
        self._tools_by_name: dict[str, Callable] = {}  # Maps function_name -> callable for execution
    
    def _augment_docstring(self, docstring: str) -> str:
        """
        Use LLM to improve and expand a tool's docstring.
        
        Takes a basic docstring and returns an enhanced version with:
        - Clearer description of what the tool does
        - Better formatted parameters and return values
        - Usage examples and edge cases
        
        Args:
            docstring: The original docstring to augment
            
        Returns:
            An improved, more detailed docstring
        """
        if not docstring.strip():
            return "No description provided."


        # NOTE: The role description of a technical writer below is a prompt engineering technique that is used to improve the quality of the docstring
        # Athough there are research that suggest that role description doesn't realy affect the quality of the LLM's output, it is still a useful technique
        # and it is a good [prompt engineering] technique to know.
        prompt = f"""You are a technical writer. Improve the following function docstring to be more clear, 
            comprehensive, and useful. Include:
            1. A clear concise summary
            2. Detailed description of what the function does
            3. When to use this function
            4. Any important notes or caveats

            Original docstring:
            {docstring}

            Return ONLY the improved docstring, no other text.
        """

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=500
        )
        
        return response.choices[0].message.content.strip()
    
    def _generate_queries(self, docstring: str, num_queries: int = 5) -> list[str]:
        """
        Generate synthetic example queries that would lead to using this tool.
        
        These queries are used to improve retrieval - by embedding both the tool
        description AND example queries, we increase the chances of finding the
        right tool when the user asks a related question.
        
        Args:
            docstring: The tool's docstring (ideally augmented)
            num_queries: Number of example queries to generate
            
        Returns:
            List of example natural language queries
        """
        prompt = f"""Based on the following tool description, generate {num_queries} diverse example queries 
            that a user might ask when they need this tool. Make them natural and varied.

            Tool description:
            {docstring}

            Return ONLY a JSON array of strings, like: ["query1", "query2", ...]
        """

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=300
        )
        
        try:
            import json
            queries = json.loads(response.choices[0].message.content.strip())
            return queries if isinstance(queries, list) else []
        except json.JSONDecodeError:
            # Fallback: extract queries from text
            return [response.choices[0].message.content.strip()]
    
    def _get_tool_metadata(self, func: Callable) -> ToolMetadata:
        """
        Extract metadata from a function for storage and retrieval.
        
        Args:
            func: The function to extract metadata from
            
        Returns:
            ToolMetadata object with function details
        """
        sig = inspect.signature(func)
        
        # Extract parameter info
        parameters = {}
        for name, param in sig.parameters.items():
            param_info = {"name": name}
            if param.annotation != inspect.Parameter.empty:
                param_info["type"] = str(param.annotation)
            if param.default != inspect.Parameter.empty:
                param_info["default"] = str(param.default)
            parameters[name] = param_info
        
        # Extract return type
        return_type = "Any"
        if sig.return_annotation != inspect.Signature.empty:
            return_type = str(sig.return_annotation)
        
        return ToolMetadata(
            name=func.__name__,
            description=func.__doc__ or "No description",
            signature=str(sig),
            parameters=parameters,
            return_type=return_type
        )
    
    def register_tool(
        self, func: Optional[Callable] = None, augment: bool = False
    ) -> Union[str, Callable]:
        """
        Register a function as a tool in the toolbox.

        Can be used as a decorator or called directly:
        
            @toolbox.register_tool
            def my_tool(): ...
            
            @toolbox.register_tool(augment=True)
            def my_enhanced_tool(): ...
            
            tool_id = toolbox.register_tool(some_function)

        Parameters:
        -----------
        func : Callable, optional
            The function to register as a tool. If None, returns a decorator.
        augment : bool, optional
            Whether to augment the tool docstring and generate synthetic queries
            using the configured LLM provider.
            
        Returns:
        --------
        Union[str, Callable]
            If func is provided, returns the tool ID. Otherwise returns a decorator.
        """

        def decorator(f: Callable) -> str:
            docstring = f.__doc__ or ""
            signature = str(inspect.signature(f))
            object_id = uuid.uuid4()
            object_id_str = str(object_id)

            # NOTE: Augmentation is a technique that is used to improve the quality of the tool's docstring
            # by using the LLM to enhance the tool's discoverability and retrieval this is a [memory engineering] technique
            if augment:
                # Use LLM to enhance the tool's discoverability
                augmented_docstring = self._augment_docstring(docstring)
                queries = self._generate_queries(augmented_docstring)
                
                # Create rich embedding text combining all information
                embedding_text = f"{f.__name__} {augmented_docstring} {signature} {' '.join(queries)}"
                embedding = get_embedding(embedding_text)
                
                tool_data = self._get_tool_metadata(f)
                tool_data.description = augmented_docstring  # Use augmented description

                tool_dict = {
                    "_id": object_id_str,  # Use string, not UUID object
                    "embedding": embedding,
                    "queries": queries,
                    "augmented": True,
                    **tool_data.model_dump(),
                }
            else:
                # Basic registration without augmentation
                embedding = get_embedding(f"{f.__name__} {docstring} {signature}")
                tool_data = self._get_tool_metadata(f)

                tool_dict = {
                    "_id": object_id_str,  # Use string, not UUID object
                    "embedding": embedding,
                    "augmented": False,
                    **tool_data.model_dump(),
                }

            # Store the tool in the toolbox memory for retrieval
            # The embedding enables semantic search to find relevant tools
            self.memory_manager.write_toolbox(
                f"{f.__name__} {docstring} {signature}", 
                tool_dict
            )
            
            # Keep reference to the callable for execution
            self._tools[object_id_str] = f
            self._tools_by_name[f.__name__] = f  # Also store by name for easy lookup
            return object_id_str

        if func is None:
            return decorator
        return decorator(func)


In [ ]:
import os
import getpass

# Function to securely get and set environment variables
def set_env_securely(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value


In [ ]:
set_env_securely("OPENAI_API_KEY", "OpenAI API Key: ")

In [ ]:
from openai import OpenAI

client = OpenAI()

# Initialize the Toolbox
toolbox = Toolbox(memory_manager=memory_manager, llm_client=client)

# Part 4: Context Engineering Techniques

--------


> **Context engineering** refers to the set of strategies for curating and maintaining the optimal set of tokens (information) during LLM inference, including all the other information that may land there outside of the prompts.
> 
> — *Anthropic*

While memory engineering focuses on *what to store and retrieve*, context engineering focuses on *how to manage what's in the context window right now*. This includes monitoring usage, compressing information, and providing just-in-time access to details.

## What This Section Covers

| Step | Function | Purpose |
|------|----------|---------|
| **1. Calculate Usage** | `calculate_context_usage()` | Monitor what % of the context window is used |
| **2. Summarize** | `summarise_context_window()` | Compress long content into summaries using LLM |
| **3. Compact** | `summarize_conversation()` / `summarize_and_store()` | Agent-triggered compaction when context gets long |
| **4. Just-in-Time Retrieval** | `expand_summary()` tool | Let agent expand summaries on demand |

**`Just-In-Time (JIT)`** retrieval is the process of fetching only the information needed at the exact moment the agent requires it, based on the current task, query, or reasoning step. Instead of loading pre-computed or pre-cached context upfront, the system dynamically retrieves the minimal, most relevant data on demand, ensuring efficiency and reducing context overload. In the context of agent memory JIT is a retrieval-control strategy where memory access is triggered by the agent’s current goal, query, or reasoning step. Rather than preloading large histories or the full knowledge base, the system dynamically filters, ranks, and injects only the information that materially influences the next token. This reduces context saturation, improves attention allocation, and increases reasoning fidelity.

## The Context Management Flow

```
Context built → Check usage % → Agent may compact (summarize) → Store summary with ID
                                                              ↓
Agent sees: [Summary ID: abc123] Brief description ← Agent can call expand_summary("abc123") if needed
```

This approach keeps the context lean while giving the agent access to full details when required.

In [ ]:
# Context window calculator - returns percentage used
def calculate_context_usage(context: str, model: str = "gpt-5") -> dict:
    """Calculate context window usage as percentage."""
    estimated_tokens = len(context) // 4  # ~4 chars per token
    max_tokens = MODEL_TOKEN_LIMITS.get(model, 128000)
    percentage = (estimated_tokens / max_tokens) * 100
    return {"tokens": estimated_tokens, "max": max_tokens, "percent": round(percentage, 1)}


In [ ]:
# Context summariser - calls LLM and stores summary
import uuid

def summarise_context_window(content: str, memory_manager, llm_client, model: str = "gpt-5") -> dict:
    """Summarise context window using LLM and store in summary memory."""
    summary_prompt = f"""
You are compressing an AI agent context window for later retrieval.
The content may include conversation memory, retrieved papers, entities, workflows, and prior summaries.

Produce a compact summary that preserves:
- user goal and constraints
- key facts/findings already established
- important entities (paper titles, arXiv IDs, authors)
- unresolved questions and next actions

Output 4-7 short bullet points.
Be faithful to the source, and do not add new facts.

Context window content:
{content[:3000]}
""".strip()

    response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": summary_prompt}],
        max_completion_tokens=220
    )
    summary = response.choices[0].message.content

    desc_response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Write a short label (max 12 words) for this summary:\n{summary}"}],
        max_completion_tokens=40
    )
    description = desc_response.choices[0].message.content.strip()

    summary_id = str(uuid.uuid4())[:8]
    memory_manager.write_summary(summary_id, content, summary, description)

    return {"id": summary_id, "description": description, "summary": summary}


In [ ]:
# Context offloader - replaces content with summary reference
def offload_to_summary(context: str, memory_manager, llm_client, threshold_percent: float = 80.0) -> tuple:
    """If context exceeds threshold, summarise and return compacted version."""
    usage = calculate_context_usage(context)
    
    if usage['percent'] < threshold_percent:
        return context, []  # No offload needed
    
    # Summarise the context
    result = summarise_context_window(context, memory_manager, llm_client)
    
    # Return compact reference instead of full content
    compact = f"[Summary ID: {result['id']}] {result['description']}"
    return compact, [result]


### Step 1: Summary Tools & Conversation Compaction

Below we register the `expand_summary` and `summarize_and_store` functions as tools the agent can call.

#### Design Logic: Why Mark Instead of Delete?

When conversation history grows large, we need to reduce context window usage. We had two choices:

| Approach | Pros | Cons |
|----------|------|------|
| **Delete summarized messages** | Simple, immediate space savings | Permanent data loss, can't audit or recover |
| **Mark as summarized (our choice)** | Preserves history, reversible, auditable | Slightly more complex queries |

**Our intuition:** Memory should be *compressed*, or *forgotten* not *erased*. By marking messages with a `summary_id` instead of deleting them:

1. **Full history is preserved** — Original messages remain in the database for auditing, debugging, or reprocessing
2. **Linkage is maintained** — Each summary knows which messages it represents (via `summary_id`)
3. **Reversible** — If a summary is deleted, you could "unsummarize" by clearing the `summary_id`

#### The Flow

```
Thread has 50 messages → Context too large → summarize_conversation(thread_id)
                                                    ↓
                        1. Read unsummarized messages
                        2. LLM summarizes them
                        3. Store summary with unique ID
                        4. UPDATE messages SET summary_id = 'abc123'
                                                    ↓
                        Next read: Only new messages appear + Summary ID reference
```

This is a form of **log compaction** — a pattern borrowed from databases and message queues where old entries are compressed but not lost.

In [ ]:
# Summary tools for the agent
@toolbox.register_tool(augment=True)
def expand_summary(summary_id: str) -> str:
    """Expand a summary reference to full content, including the original conversation
    messages that were compacted into it. Use when you need more details from a [Summary ID: xxx] reference."""
    summary_text = memory_manager.read_summary_memory(summary_id)
    
    # Also retrieve the original conversation messages linked to this summary
    original_msgs = memory_manager.get_messages_by_summary_id(summary_id)
    if original_msgs:
        lines = [f"[{m['role']}] {m['content']}" for m in original_msgs]
        return f"Summary:\n{summary_text}\n\nOriginal messages ({len(original_msgs)}):\n" + "\n".join(lines)
    return summary_text

@toolbox.register_tool(augment=True)
def summarize_and_store(text: str) -> str:
    """Summarize a long text block and store it. Returns [Summary ID: ...] for later expansion."""
    result = summarise_context_window(text, memory_manager, client)
    return f"Stored as [Summary ID: {result['id']}] {result['description']}"

@toolbox.register_tool(augment=True)
def summarize_conversation(thread_id: str) -> str:
    """
    Summarize unsummarized conversation units for a thread and mark those units with summary_id.
    Use this when conversation memory becomes long and you need context compaction.
    """
    unsummarized = memory_manager.get_unsummarized_messages(thread_id, limit=200)
    if not unsummarized:
        return "No unsummarized conversation units found."

    full_text = "\n".join([f"[{m['role']}] {m['content']}" for m in unsummarized])
    result = summarise_context_window(full_text, memory_manager, client)

    message_ids = [m["id"] for m in unsummarized]
    memory_manager.mark_as_summarized(thread_id, result['id'], message_ids=message_ids)

    return f"Conversation summarized as [Summary ID: {result['id']}] {result['description']}"

# Part 5: Web Access with Tavily

--------

This section demonstrates how to create an **agentic tool** that the LLM can call to search the web. 

We use [Tavily](https://tavily.com/), an AI-optimized search API designed for LLM applications.

## What This Section Does

1. **Initialize the Tavily client** — Set up the search API with an API key
2. **Register `search_tavily` as a tool** — Use `@toolbox.register_tool(augment=True)` to make it discoverable
3. **Implement the search-and-store pattern** — Results are automatically written to knowledge base memory
4. **Test tool retrieval** — Verify the tool can be found via semantic search

## The Search-and-Store Pattern

One thing to note is that not only do we get external context that is not available to the Agent at execution, but we persists this to the knowledge base memory and the Agent can reuse this information in subsequent iteration.
When the agent calls `search_tavily()`, it doesn't just return results—it **persists them to the knowledge base**:

```
Agent calls search_tavily("latest AI news")
       ↓
Tavily API returns results
       ↓
Each result is written to knowledge_base_vs with metadata (title, URL, timestamp)
       ↓
Future queries can retrieve this information without searching again
```

This pattern means the agent **learns** from its searches. Information discovered once becomes part of the agent's long-term memory, available for future conversations without additional API calls.

In [ ]:
set_env_securely("TAVILY_API_KEY", "Tavily API Key: ")

In [ ]:
from tavily import TavilyClient
from datetime import datetime

# Don't forget to set your API key!
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@toolbox.register_tool(augment=True)
def search_tavily(query: str, max_results: int = 5):
    """
    Use this function to search the web and store the results in the knowledge base.
    """
    response = tavily_client.search(query=query, max_results=max_results)
    results = response.get("results", [])

    # Write each result to the knowledge base
    for result in results:
        # Create the text content to embed
        text = f"Title: {result.get('title', '')}\nContent: {result.get('content', '')}\nURL: {result.get('url', '')}"
        
        # Create metadata
        metadata = {
            "title": result.get("title", ""),
            "url": result.get("url", ""),
            "score": result.get("score", 0),
            "source_type": "tavily_search",
            "query": query,
            "timestamp": datetime.now().isoformat()
        }
        
        # Write to knowledge base
        memory_manager.write_knowledge_base(text, metadata)

    return results

In [ ]:
import pprint
retreived_tools = memory_manager.read_toolbox("Search the internet")
pprint.pprint(retreived_tools)

# Part 6: Agent Execution

--------


This is where everything comes together. We build a complete **turn-level agent harness** that integrates all the memory types, context engineering, and tool calling we've implemented.

## What This Section Contains

| Component | Purpose |
|-----------|---------|
| `AGENT_SYSTEM_PROMPT` | Instructions telling the LLM how to use memory and tools |
| `execute_tool()` | Looks up and executes tools from the toolbox by name |
| `call_openai_chat()` | Wrapper for OpenAI Chat Completions API with tool support |
| `call_agent()` | Turn-level harness for one agent run (build context, run tool-call loop, persist results) |


In [ ]:
import json as json_lib

client = OpenAI()

# Persistent context-window tracker — survives across call_agent() invocations
context_size_history = []  # list of (run_label, iteration, estimated_tokens)

# ==================== SYSTEM PROMPT ====================
AGENT_SYSTEM_PROMPT = """
# System Instructions
You are a Research Paper Assistant with access to memory and tools.

IMPORTANT: The user's input contains CONTEXT retrieved from multiple memory systems.
Each memory section has a Purpose and When-to-use guide — follow them.

## Memory Priority Order
1. **Conversation Memory** — check what the user already asked and what you already answered.
2. **Knowledge Base Memory** — cite facts from stored papers/documents before searching externally.
3. **Entity Memory** — resolve named references ("that author", "the system") from here.
4. **Workflow Memory** — reuse proven tool sequences for similar past queries.
5. **Summary Memory** — expand a summary ID only when you need specific details from older context.

## Tool Output Handling
Tool call outputs are logged to a Tool Log table and replaced with compact references in context.
The preview in each [Tool Log ...] reference contains enough to reason about the result.
If you need the full output, it can be retrieved from the database — but prefer working with
the preview and the knowledge base (where search results are also stored).

## Context Management
If conversation memory is getting long or repetitive, call summarize_conversation(thread_id) to compact it.
Use summarization tools at your discretion when they improve context quality.

When answering:
1. FIRST, use the context provided in the input
2. Expand summary IDs just-in-time when needed
3. Use external search tools only if memory context is insufficient
4. Keep responses evidence-based and aligned with retrieved research context
"""

def execute_tool(tool_name: str, tool_args: dict) -> str:
    """Execute a tool by looking it up in the toolbox."""

    if tool_name not in toolbox._tools_by_name:
        return f"Error: Tool '{tool_name}' not found"

    return str(toolbox._tools_by_name[tool_name](**tool_args) or "Done")

# ==================== OPENAI CHAT FUNCTION ====================
def call_openai_chat(messages: list, tools: list = None, model: str = "gpt-5"):
    """Call OpenAI Chat Completions API with tools."""
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    return client.chat.completions.create(**kwargs)

## Step 1: The Turn-Level Agent Run Flow

```
1. BUILD CONTEXT (programmatic)
   ├── Read conversational memory (unsummarized chat units)
   ├── Read knowledge base (relevant documents)
   ├── Read workflow memory (past action patterns)
   ├── Read entity memory (people, places, systems)
   └── Read summary context (available summary IDs + descriptions)

2. GET TOOLS (programmatic)
   └── Retrieve semantically relevant tools from toolbox

3. STORE USER MESSAGE (programmatic)
   └── Persist the user message + best-effort entity extraction

4. WITHIN-RUN TOOL-CALL LOOP (up to max_iterations and within max_execution_time_s)
   ├── Call LLM with context + tool schemas
   ├── If tool calls → execute tools and append tool outputs
   ├── If tools changed memory (search/compaction) → rebuild context for the next iteration
   └── If no tool calls → finalize answer

5. GUARDED STOP
   └── If iteration/time budget is hit → force a final best-effort answer (no tools)

6. SAVE RESULTS (programmatic)
   ├── Write workflow (if tools were used)
   ├── Best-effort entity extraction on final answer
   └── Store assistant response in conversational memory
```

## Key Design Decisions

- **Memory is loaded programmatically** so the model always starts from a consistent state.
- **Tool use is agent-triggered** but tool execution is deterministic (the harness executes exactly what was requested).
- **Context and memory engineering are first-class**: compaction/search can refresh context within the same run.
- **Guardrails matter**: iteration and time budgets prevent runaway loops; the harness can still produce a final answer.


In [ ]:
# ==================== TURN-LEVEL AGENT HARNESS (ONE RUN) ====================
def call_agent(query: str, thread_id: str = "1", max_iterations: int = 10, max_execution_time_s: float = 60.0) -> str:
    """Turn-level agent harness: build context, run tool-call loop, persist results.
    
    Appends (run_label, iteration, tokens) to the global context_size_history list
    so context growth can be visualised across multiple runs.
    """
    thread_id = str(thread_id)
    steps = []
    run_label = f"Run {len(set(r for r, _, _ in context_size_history)) + 1}"

    import time

    start_time = time.time()
    timed_out = False

    # 1. Build context from memory
    print("\n" + "="*50)
    print("🧠 BUILDING CONTEXT...")

    def build_context() -> str:
        """Rebuild the full context from the current memory state."""
        ctx = f"# Question\n{query}\n\n"
        ctx += memory_manager.read_conversational_memory(thread_id) + "\n\n"
        ctx += memory_manager.read_knowledge_base(query) + "\n\n"
        ctx += memory_manager.read_workflow(query) + "\n\n"
        ctx += memory_manager.read_entity(query) + "\n\n"
        ctx += memory_manager.read_summary_context(query) + "\n\n"  # IDs + descriptions only
        return ctx

    context = build_context()

    print("====CONTEXT WINDOW=====\n")
    print(context)

    # 2. Check context usage (agent decides whether to summarize via tools)
    usage = calculate_context_usage(context)
    print(f"📊 Context: {usage['percent']}% ({usage['tokens']}/{usage['max']} tokens)")
    if usage['percent'] > 80:
        print("⚠️ Context >80% - agent may call summarize_conversation(thread_id) for compaction.")

    # 3. Get tools
    dynamic_tools = memory_manager.read_toolbox(query, k=5)

    # Ensure summary tools are available for discretionary compaction/JIT expansion
    summary_tool_candidates = memory_manager.read_toolbox(
        "summarize conversation compact context expand summary memory", k=5
    )
    must_have = {"expand_summary", "summarize_conversation", "summarize_and_store"}
    existing = {t.get("function", {}).get("name") for t in dynamic_tools}

    for tool in summary_tool_candidates:
        name = tool.get("function", {}).get("name")
        if name in must_have and name not in existing:
            dynamic_tools.append(tool)
            existing.add(name)

    print(f"🔧 Tools: {[t['function']['name'] for t in dynamic_tools]}")

    # 4. Store user message & extract entities
    memory_manager.write_conversational_memory(query, "user", thread_id)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=query)
    except:
        pass

    # 5. Within-run tool-call loop
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}, {"role": "user", "content": context}]
    final_answer = ""

    # Estimate tool schema tokens (sent with every API call)
    tool_schema_tokens = len(json_lib.dumps(dynamic_tools)) // 4 if dynamic_tools else 0

    print("\n🤖 TOOL-CALL LOOP")
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")

        # Record context window size to the global tracker (messages + tool schemas)
        total_chars = sum(len(m.get("content", "") or "") for m in messages)
        est_tokens = (total_chars // 4) + tool_schema_tokens
        context_size_history.append((run_label, iteration + 1, est_tokens))

        if max_execution_time_s is not None:
            elapsed = time.time() - start_time
            if elapsed > max_execution_time_s:
                timed_out = True
                print(f"\n⏱️ Time limit reached ({elapsed:.1f}s > {max_execution_time_s:.1f}s). Finalizing...")
                break

        response = call_openai_chat(messages, tools=dynamic_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]})

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                raw_args = tc.function.arguments or "{}"
                try:
                    tool_args = json_lib.loads(raw_args)
                except Exception as e:
                    result = f"Error: invalid JSON tool arguments for {tool_name}: {e}. Raw: {raw_args}"
                    print(f"🛠️ {tool_name}(<invalid args>)")
                    steps.append(f"{tool_name}(<invalid args>) → failed")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                    continue

                if not isinstance(tool_args, dict):
                    result = f"Error: tool arguments for {tool_name} must be a JSON object. Got {type(tool_args).__name__}."
                    print(f"🛠️ {tool_name}(<non-object args>)")
                    steps.append(f"{tool_name}(<non-object args>) → failed")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                    continue

                # Ensure conversation compaction always targets the active thread.
                if tool_name == "summarize_conversation":
                    tool_args["thread_id"] = thread_id

                args_display = {k: (v[:50] + '...' if isinstance(v, str) and len(v) > 50 else v)
                               for k, v in tool_args.items()}
                print(f"🛠️ {tool_name}({args_display})")

                if max_execution_time_s is not None:
                    elapsed = time.time() - start_time
                    if elapsed > max_execution_time_s:
                        timed_out = True
                        result = f"Error: time limit reached before executing tool {tool_name}."
                        steps.append(f"{tool_name}({args_display}) → failed")
                        print(f"   → {result}")
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                        break

                try:
                    result = execute_tool(tool_name, tool_args)
                    steps.append(f"{tool_name}({args_display}) → success")
                except Exception as e:
                    result = f"Error: {e}"
                    steps.append(f"{tool_name}({args_display}) → failed")

                print(f"   → {result[:200]}...")

                # Offload tool output to TOOL_LOG table (experimental memory).
                # Full output is persisted in the DB; only a compact reference
                # stays in the messages list to keep the context window lean.
                compact_result = memory_manager.write_tool_log(
                    thread_id, tc.id, tool_name, raw_args, str(result)
                )
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": compact_result})

                # If tools changed memory state, refresh context for the next iteration.
                if tool_name in {"search_tavily", "summarize_conversation", "summarize_and_store"}:
                    context = build_context()
                    if len(messages) >= 2 and messages[1].get("role") == "user":
                        messages[1]["content"] = context
                    usage = calculate_context_usage(context)
                    print(f"   Refreshed context: {usage['percent']}% ({usage['tokens']}/{usage['max']} tokens)")

            if timed_out:
                break
        else:
            final_answer = msg.content or ""
            print(f"\n✅ DONE ({len(steps)} tool calls)")
            break

    if not final_answer:
        reason = "time limit" if timed_out else "iteration limit"
        print(f"\n⚠️ Stopped due to {reason}. Generating best-effort final answer (no tools)...")
        try:
            final_messages = messages + [{"role": "user", "content": "Finalize your answer using the context and tool outputs so far. Do not call tools."}]
            final_resp = call_openai_chat(final_messages, tools=None)
            final_answer = final_resp.choices[0].message.content or ""
        except Exception as e:
            final_answer = f"Error: unable to finalize answer: {e}"

    # 6. Save workflow & entities
    if steps:
        memory_manager.write_workflow(query, steps, final_answer)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=final_answer)
    except:
        pass
    memory_manager.write_conversational_memory(final_answer, "assistant", thread_id)

    print("\n" + "="*50 + f"\n💬 ANSWER:\n{final_answer}\n" + "="*50)
    return final_answer

In [ ]:
call_agent("What was my first question to you", thread_id="0022")

In [ ]:
import matplotlib.pyplot as plt

if context_size_history:
    tokens = [t for _, _, t in context_size_history]

    plt.figure(figsize=(8, 3))
    plt.plot(range(1, len(tokens) + 1), tokens, marker="o")
    plt.xlabel("Global Iteration (across all runs)")
    plt.ylabel("Estimated Tokens")
    plt.title("Context Window Size Over Agent Iterations")
    plt.tight_layout()
    plt.show()
else:
    print("No iterations recorded — run call_agent() first.")

## Step 2: Baseline — Agent Without Context Engineering

To appreciate the impact of the memory and context engineering techniques we've built, it helps to see what happens **without them**.

`call_agent_naive` is a stripped-down agent harness that deliberately removes three key optimisations:

| Technique Removed | What Happens Instead | Effect on Context Window |
|---|---|---|
| **Tool output offloading** (`write_tool_log`) | Full raw tool outputs stay in the `messages` list | Each tool call adds thousands of tokens (e.g. a web search returns ~2-4k tokens of results) |
| **Summarisation tools** (`summarize_conversation`, `summarize_and_store`) | Excluded from the tool list — the agent has no way to compact context | Context only grows, never shrinks |
| **Context refresh after search** | No rebuild from memory after tool calls | Stale + bloated context persists across iterations |
| **Memory-backed context rebuild** | Messages persist as one flat list across calls | No separation of concerns — everything accumulates |

### Why This Matters

In a real agent loop, the LLM is called **once per iteration** with the full `messages` list. Without offloading, every tool output ever produced sits in that list. After just 3 web searches, the context could grow by 10,000+ tokens — consuming budget that could be used for reasoning.

The comparison chart below plots both approaches on the same axis so you can see the divergence.

In [ ]:
# Separate tracker for the naive agent
naive_context_size_history = []
# Persistent messages per thread — simulates no context management across runs
_naive_messages_by_thread = {}

def call_agent_naive(query: str, thread_id: str = "naive_1", dynamic_tools_override: list = None, max_iterations: int = 10, max_execution_time_s: float = 60.0) -> str:
    """Naive agent harness — NO context engineering.
    
    Differences from call_agent:
    1. Full raw tool outputs stay in messages (no write_tool_log offloading)
    2. No summarisation tools available (agent cannot compact context)
    3. No context refresh after memory-mutating tools
    4. Messages persist across calls — context only grows, never shrinks
    5. No memory reads — conversation history IS the raw messages list
    """
    thread_id = str(thread_id)
    steps = []
    import time
    start_time = time.time()
    timed_out = False

    # Get tools — but exclude summarisation tools
    if dynamic_tools_override is not None:
        dynamic_tools = dynamic_tools_override
    else:
        dynamic_tools = memory_manager.read_toolbox(query, k=5)
    dynamic_tools = [t for t in dynamic_tools
                     if t.get("function", {}).get("name") not in
                     {"summarize_conversation", "summarize_and_store", "expand_summary"}]

    # Initialize or reuse persistent messages for this thread.
    # No memory reads — the raw messages list IS the only context.
    # This is the naive approach: everything accumulates in one flat list.
    if thread_id not in _naive_messages_by_thread:
        _naive_messages_by_thread[thread_id] = [
            {"role": "system", "content": "You are a Research Paper Assistant with access to tools."}
        ]
    messages = _naive_messages_by_thread[thread_id]

    # Just append the raw query — no build_context(), no memory reads.
    # Prior turns, tool outputs, and assistant responses are already in messages.
    messages.append({"role": "user", "content": query})
    final_answer = ""

    # Estimate tool schema tokens (included in every API call)
    tool_schema_chars = len(json_lib.dumps(dynamic_tools)) if dynamic_tools else 0
    tool_schema_tokens = tool_schema_chars // 4

    for iteration in range(max_iterations):
        # Track context size: messages + tool schemas
        msg_chars = sum(len(m.get("content", "") or "") for m in messages)
        naive_context_size_history.append((msg_chars // 4) + tool_schema_tokens)

        if max_execution_time_s and (time.time() - start_time) > max_execution_time_s:
            timed_out = True
            break

        response = call_openai_chat(messages, tools=dynamic_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]})
            for tc in msg.tool_calls:
                tool_args = json_lib.loads(tc.function.arguments or "{}")
                try:
                    result = execute_tool(tc.function.name, tool_args)
                    steps.append(f"{tc.function.name} → success")
                except Exception as e:
                    result = f"Error: {e}"
                    steps.append(f"{tc.function.name} → failed")

                # KEY DIFFERENCE: raw tool output goes straight into messages (no offloading)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
                # KEY DIFFERENCE: no context refresh
        else:
            final_answer = msg.content or ""
            break

    if not final_answer:
        try:
            messages.append({"role": "user", "content": "Finalize your answer. Do not call tools."})
            final_answer = call_openai_chat(messages, tools=None).choices[0].message.content or ""
        except Exception as e:
            final_answer = f"Error: {e}"

    # Append assistant answer to persistent messages (it stays for the next call)
    messages.append({"role": "assistant", "content": final_answer})
    print(f"✅ Naive agent done ({len(steps)} tool calls, {len(messages)} messages in context)")
    return final_answer

In [ ]:
import uuid

# Reset all trackers for a clean comparison
context_size_history.clear()
naive_context_size_history.clear()
_naive_messages_by_thread.clear()

# Generate unique thread IDs for isolation
eng_thread = str(uuid.uuid4())[:8]
naive_thread = str(uuid.uuid4())[:8]

# Five progressive queries that build on each other — tests memory continuity
queries = [
    "Search for recent papers on AI agent memory published in 2026",
    "Pick the 3rd paper from the list and give me the key takeaways",
    "What other viewpoints or approaches might that paper have missed?",
    "Summarize everything we've discussed so far",
    "What was the first question I asked in this conversation?",
]

for i, q in enumerate(queries, 1):
    print("=" * 60)
    print(f"QUERY {i}/5 — WITH CONTEXT ENGINEERING (thread: {eng_thread})")
    print(f"  >> {q}")
    print("=" * 60)
    call_agent(q, thread_id=eng_thread)
    print("\n")

for i, q in enumerate(queries, 1):
    print("=" * 60)
    print(f"QUERY {i}/5 — NAIVE / NO CONTEXT ENGINEERING (thread: {naive_thread})")
    print(f"  >> {q}")
    print("=" * 60)
    call_agent_naive(q, thread_id=naive_thread)
    print("\n")

In [ ]:
import matplotlib.pyplot as plt

eng_tokens = [t for _, _, t in context_size_history]
naive_tokens = naive_context_size_history

plt.figure(figsize=(9, 4))
if eng_tokens:
    plt.plot(range(1, len(eng_tokens) + 1), eng_tokens, marker="o", label="With Context/Memory Engineering")
if naive_tokens:
    plt.plot(range(1, len(naive_tokens) + 1), naive_tokens, marker="s", label="Naive (no offloading/summarisation)")
plt.xlabel("Iteration")
plt.ylabel("Estimated Tokens")
plt.title("Context Window Growth: Engineered vs Naive Agent")
plt.legend()
plt.tight_layout()
plt.show()

## Key Learning for AI Developers: Agent Run, Tool-Call Loop, and Memory-Based Harness

In OpenAI-style framing:
- An **agent run** (one user turn handled) is what `call_agent(...)` executes.
- Within a run, the **tool-call loop** repeats: model reasoning → optional tool calls → harness executes tools → model observes results → repeat until a final answer.

An **agent harness** is the runtime scaffolding around that loop. In this notebook, it is a **memory-based agent harness** where:
- context is assembled from multiple memory types each run
- tools are discovered and executed within the run
- outputs are written back into memory for future runs
- summaries compact context while preserving continuity

The key discipline is **context and memory engineering**:
- decide what should be stored, retrieved, summarized, and refreshed
- keep context windows relevant, not just large
- treat memory as an evolving system that improves agent reliability over time

The practical takeaway: strong agents are not just model prompts. They are run + harness systems, and memory engineering is the control layer that makes them reliable, stateful, and scalable.


## Step 3: LLM-as-a-Judge — Response Quality Evaluation

We've seen the **context window efficiency** difference between the two agents. But does better context engineering actually produce **better answers**?

To find out, we use the **LLM-as-a-Judge** pattern: a separate LLM call evaluates both agent responses against the original query and picks a winner. This is a widely used technique for automated evaluation when ground-truth labels aren't available.

| What the Judge Sees | What It Decides |
|---|---|
| The user query | Which response is more **accurate, complete, and relevant** |
| Response A (memory-engineered agent) | A preference: **A**, **B**, or **Tie** |
| Response B (naive agent) | A short explanation of its reasoning |

> **Why a warmup phase?** The memory agent's advantage is **cumulative** — it stores conversational memory, entities, and workflows across turns while managing context size. On a brand-new conversation, both agents perform similarly. We first run 5 warmup queries to build up conversation state, then evaluate on queries that specifically test **recall, continuity, and synthesis** — the capabilities that memory engineering enables.

In [ ]:
# ── Warmup phase: build up conversation history so the memory agent has state to leverage ──
eval_thread_eng = str(uuid.uuid4())[:8]
eval_thread_naive = str(uuid.uuid4())[:8]

warmup_queries = [
    "Search for recent papers on AI agent memory published in 2026",
    "Pick the 2nd paper from the list and give me the key takeaways",
    "What other approaches might that paper have missed?",
    "Search for papers on context window management in LLM agents",
    "Compare the findings from the two searches we did",
]

print("🔄 WARMUP — building conversation history on both agents...\n")
for i, q in enumerate(warmup_queries, 1):
    print(f"  Warmup {i}/{len(warmup_queries)}: {q[:60]}...")
    call_agent(q, thread_id=eval_thread_eng)
    call_agent_naive(q, thread_id=eval_thread_naive)

# ── Evaluation phase: these queries test memory recall and continuity ──
eval_queries = [
    "What was the very first paper we discussed and what were its key points?",
    "Summarize the full arc of our conversation so far",
    "Based on everything we've discussed, what research gap would you recommend exploring next?",
]

eval_results = []

print(f"\n{'='*60}\n📋 EVALUATION — collecting response pairs for judging\n{'='*60}")
for q in eval_queries:
    print(f"\nEVAL: {q}")
    print("  ▶ Memory-engineered agent...")
    eng_resp = call_agent(q, thread_id=eval_thread_eng)
    print("  ▶ Naive agent...")
    naive_resp = call_agent_naive(q, thread_id=eval_thread_naive)
    eval_results.append((q, eng_resp, naive_resp))

print(f"\n✅ Collected {len(eval_results)} response pairs for judging.")

In [ ]:
JUDGE_PROMPT = """You are an impartial judge evaluating two AI assistant responses to a user query.

**User Query:** {query}

**Response A (Agent A):**
{response_a}

**Response B (Agent B):**
{response_b}

Evaluate both responses on:
1. **Accuracy** — Are the facts correct and claims well-supported?
2. **Completeness** — Does the response fully address the query?
3. **Relevance** — Does it stay on-topic and use context appropriately?
4. **Coherence** — Is it well-structured and easy to follow?

Reply with EXACTLY this JSON format (no other text):
{{"winner": "A" or "B" or "Tie", "reason": "one sentence explanation"}}"""


def judge_responses(query, response_a, response_b):
    """Use the LLM to judge which response is better."""
    resp = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            query=query, response_a=response_a, response_b=response_b
        )}],
    )
    return json_lib.loads(resp.choices[0].message.content)


# Run the judge on each response pair
judgments = []
for query, eng_resp, naive_resp in eval_results:
    verdict = judge_responses(query, eng_resp, naive_resp)
    verdict["query"] = query
    judgments.append(verdict)
    label = {"A": "Memory Agent ✅", "B": "Naive Agent", "Tie": "Tie 🤝"}
    print(f"Query: {query[:60]}...")
    print(f"  Winner: {label.get(verdict['winner'], verdict['winner'])}")
    print(f"  Reason: {verdict['reason']}\n")

In [ ]:
# Visualize judge results
wins = {"Memory Agent": 0, "Naive Agent": 0, "Tie": 0}
for j in judgments:
    if j["winner"] == "A":
        wins["Memory Agent"] += 1
    elif j["winner"] == "B":
        wins["Naive Agent"] += 1
    else:
        wins["Tie"] += 1

colors = {"Memory Agent": "#4CAF50", "Naive Agent": "#F44336", "Tie": "#9E9E9E"}

plt.figure(figsize=(6, 3))
bars = plt.bar(wins.keys(), wins.values(), color=[colors[k] for k in wins])
plt.ylabel("Queries Won")
plt.title("LLM Judge Preference: Memory Agent vs Naive Agent")
for bar, count in zip(bars, wins.values()):
    if count > 0:
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 str(count), ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nMemory Agent wins {wins['Memory Agent']}/{len(judgments)} queries.")